In [28]:
from typing import TypedDict
from pydantic import BaseModel
from langgraph.graph import  StateGraph, START, END


In [35]:
from typing import TypedDict


class ClassifyIntent(TypedDict):
    input_text: str
    category: str


In [36]:
def classify(state: ClassifyIntent) -> ClassifyIntent:
    text = state["input_text"].lower()

    technical_words = ["python", "code", "bug", "error", "java"]

    category = (
        "technical"
        if any(word in text for word in technical_words)
        else "general"
    )

    return {
        **state,
        "category": category,
    }


In [37]:
def technical_node(state: ClassifyIntent) -> ClassifyIntent:
    print("Running technical node")
    return state


def general_node(state: ClassifyIntent) -> ClassifyIntent:
    print("Running general node")
    return state


def category_route(state: ClassifyIntent) -> str:
    print("Running category route")

    if state["category"] == "technical":
        return "technical"

    return "general"

In [38]:
graph = StateGraph(ClassifyIntent)

graph.add_node("classify", classify)
graph.add_node("technical_node", technical_node)
graph.add_node("general_node", general_node)

graph.add_edge(START, "classify")

# Map route results to actual node names
graph.add_conditional_edges(
    "classify",
    category_route,
    {
        "technical": "technical_node",
        "general": "general_node",
    },
)

graph.add_edge("technical_node", END)
graph.add_edge("general_node", END)

workflow = graph.compile()

In [40]:
input_msg = { "input_text": "I have a Python bug"}

res = workflow.invoke(input_msg)
print(res)

Running category route
Running technical node
{'input_text': 'I have a Python bug', 'category': 'technical'}


In [41]:
input_msg_2 = {
    "input_text": "I have a Python bug",
    "category": "technical"
}


res_2 = workflow.invoke(input_msg_2)
print(res_2)

Running category route
Running technical node
{'input_text': 'I have a Python bug', 'category': 'technical'}
